In [ ]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier



In [ ]:
data=load_breast_cancer(as_frame=True)
df=data.frame
df.head()

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

X_train ,X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)


In [ ]:
rf = RandomForestClassifier()
rf.fit(X_train,y_train)
rf_pred = rf.predict(X_test)
print("Accuracy Score : ",accuracy_score(y_test,rf_pred))
print("Classification Report:",classification_report(y_test,rf_pred))

In [ ]:
xg = XGBClassifier()
xg.fit(X_train,y_train)
xg_pred = xg.predict(X_test)

print("accuracy Score: ", accuracy_score(y_test,xg_pred))
print("Classification Report:",classification_report(y_test,xg_pred))

In [ ]:
lg = LGBMClassifier()
lg.fit(X_train , y_train)
lg_pred = lg.predict(X_test)
print("accuarcy Score : ", accuracy_score(y_test, lg_pred))
print("Classification Report:",classification_report(y_test,lg_pred))

In [ ]:
comparison = pd.DataFrame({
    "model": [
        "Random Forest",
        "XGBoost",
        "LightGBM"
    ],
    "accuracy": [
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, xg_pred),
        accuracy_score(y_test, lg_pred)
    ]
})
print(comparison)

Date -- 02-07-2026
Training

cm = confusion_matrix(
    y_test,
    rf_pred
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm
)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score

rf_prob = rf.predict_proba(X_test)[:, 1]
roc_score = roc_auc_score(y_test,rf_prob)
print('ROC AUC Score: ',roc_score)

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_test, rf_prob)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_score:.2f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()


In [ ]:
!pip install optuna
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier
import optuna


In [ ]:
model = XGBClassifier(random_state=42)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')

param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 4, 5]
}

grid_search = GridSearchCV(model, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)
print("Best parameters")
print(grid_search.best_params_)
best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')

In [ ]:
random_param_grid = {
    'n_estimators': [100, 200, 30],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 4, 5]
}
random_search = RandomizedSearchCV(model, random_param_grid, cv=5, scoring='accuracy')
random_search.fit(X_train, y_train)
print("Best parameters")
print(random_search.best_params_)
best_random_model = random_search.best_estimator_
prediction = best_random_model.predict(X_test)
accuracy = accuracy_score(y_test, prediction)
print(f'Accuracy: {accuracy}')

In [ ]:
def objective(trial):
  model=XGBClassifier(
      n_estimators=trial.suggest_int('n_estimators',100,300),
      learning_rate=trial.suggest_float('learning_rate',0.01,0.2),
      max_depth=trial.suggest_int('max_depth',3,7),
      random_state=42
  )

  model.fit(X_train,y_train)
  prediction=model.predict(X_test)
  return accuracy_score(y_test,prediction)

study=optuna.create_study(direction='maximize')
study.optimize(objective,n_trials=20)
print("Best Parameters: ",study.best_params)
best_model=XGBClassifier(**study.best_params,random_state=42)
best_model.fit(X_train,y_train)
prediction=best_model.predict(X_test)
print("Best Model Accuracy :",accuracy_score(y_test,prediction))

In [ ]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_sample, y_sample = ros.fit_resample(X_train, y_train)

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)
X_sample, y_sample = rus.fit_resample(X_train, y_train)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier(class_weight="balanced")
model.fit(X_train, y_train)

#xgb

ratio = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
model = XGBClassifier(scale_pos_weight=ratio,random_state=42)
model.fit(X_train, y_train)

In [ ]:
prob = best_model.predict_proba(X_test)[:, 1]
thresholds=0.40
prediction=(prob>=thresholds).astype(int)

for threshold in [0.5,0.4,0.2]:
  predictions=(prob>=threshold).astype(int)
  print("Threshold: ",threshold)
  print(classification_report(y_test,predictions))

In [ ]:
#summary plot
import shap
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test,feature_names=X.columns)

In [ ]:
#Force plot
shap.initjs()
shap.force_plot(explainer.expected_value,
                shap_values[0],
                X_test.iloc[0],
                feature_names=X.columns)